# THUẬT TOÁN YEN'S K-SHORTEST PATH - BỘ DỮ LIỆU TEST TỪ FILE EXCEL
Mã nguồn cơ bản, chia nhỏ từng phần, đọc toàn bộ mạng lưới giao thông (Nodes, Directed Edges, OD Pairs) từ file `Du_lieu_test.xlsx` để tìm tuyến đường Top 1 tối ưu nhất.

### Phần 1: Khai báo thư viện & Tùy chọn các cặp tuyến đường cần tìm

In [1]:
import math
import heapq
import json
import pandas as pd

# Đường dẫn file dữ liệu Excel
FILE_EXCEL = 'Du_lieu_test.xlsx'

# Đọc danh sách các cặp OD từ sheet OD_Pairs
df_od = pd.read_excel(FILE_EXCEL, sheet_name='OD_Pairs')

# Chọn 5 cặp tuyến đường theo mã OD yêu cầu
OD_IDS = ("OD06", "OD11", "OD14", "OD25", "OD28")
df_od_5 = df_od.set_index("od_id").loc[list(OD_IDS)].reset_index()
print("5 cặp tuyến đường cần tìm:")
for _, r in df_od_5.iterrows():
    print(f"  • [{r['od_id']}] {r['origin_road']} -> {r['destination_road']} (Node {r['source']} -> {r['target']})")

5 cặp tuyến đường cần tìm:
  • [OD06] Xô Viết Nghệ Tĩnh -> Nguyễn Gia Trí (Node 5778363087 -> 366400308)
  • [OD11] Xô Viết Nghệ Tĩnh -> Ung Văn Khiêm (Node 366467615 -> 366414641)
  • [OD14] Ung Văn Khiêm -> Xô Viết Nghệ Tĩnh (Node 366414641 -> 13900826831)
  • [OD25] Ung Văn Khiêm -> Điện Biên Phủ (Node 13269798556 -> 2022539300)
  • [OD28] Ung Văn Khiêm -> Nguyễn Gia Trí (Node 1276696904 -> 1276696942)


### Phần 2: Đọc dữ liệu Nodes và Directed_Edges từ file Excel

In [2]:
# 1. Đọc dữ liệu Nodes và Directed_Edges từ Excel
df_nodes = pd.read_excel(FILE_EXCEL, sheet_name='Nodes')
df_edges = pd.read_excel(FILE_EXCEL, sheet_name='Directed_Edges')

# Tạo từ điển tọa độ theo node_id
toa_do = {int(r['node_id']): (float(r['lat']), float(r['lon'])) for _, r in df_nodes.iterrows()}

# 2. Cấu hình bảng độ trễ khi rẽ (giây)
BANG_DO_TRE = {
    'di_thang': 3.0,
    're_phai': 8.0,
    're_trai': 35.0,
    'quay_dau': 75.0,
    're_trai_tu_hem_ra_truc_chinh_phat_them': 30.0,
    'phat_hem_cut': 300.0
}

print(f"Đã nạp {len(toa_do)} nodes và {len(df_edges)} cung đường có hướng từ file Excel.")

Đã nạp 4380 nodes và 8315 cung đường có hướng từ file Excel.


### Phần 3: Các hàm tính khoảng cách và độ trễ góc rẽ

In [3]:
def tinh_khoang_cach(u, v):
    lat1, lon1 = toa_do[u]
    lat2, lon2 = toa_do[v]
    d_lat, d_lon = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(d_lat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(d_lon/2)**2
    return 6371000.0 * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def tinh_do_tre_re(goc_truoc, goc_hien_tai, hem_truoc=False, hem_hien_tai=False):
    if goc_truoc is None:
        return 0.0, 'Bắt đầu'
    delta = (goc_hien_tai - goc_truoc + 180) % 360 - 180
    if -35.0 <= delta <= 35.0:
        return BANG_DO_TRE['di_thang'], 'Đi thẳng'
    elif 35.0 < delta <= 135.0:
        return BANG_DO_TRE['re_phai'], 'Rẽ phải'
    elif -135.0 <= delta < -35.0:
        tre = BANG_DO_TRE['re_trai']
        if hem_truoc and not hem_hien_tai:
            tre += BANG_DO_TRE['re_trai_tu_hem_ra_truc_chinh_phat_them']
        return tre, 'Rẽ trái'
    else:
        return BANG_DO_TRE['quay_dau'], 'Quay đầu'

### Phần 4: Xây dựng đồ thị có hướng từ bảng Directed_Edges

In [4]:
do_thi = {nid: [] for nid in toa_do}
thong_tin_canh = {}

for _, r in df_edges.iterrows():
    u, v = int(r['u']), int(r['v'])
    if u not in toa_do or v not in toa_do:
        continue
    cid = str(r['arc_id'])
    do_dai = float(r['length_m'])
    # Dùng peak_time_s cho giờ cao điểm theo hướng dẫn của file test
    thoi_gian = float(r['peak_time_s']) if pd.notna(r['peak_time_s']) else float(r['time_s'])
    la_hem = bool(r['is_alley'])
    goc = float(r['bearing_deg']) if pd.notna(r['bearing_deg']) else 0.0
    ten = str(r['name']) if pd.notna(r['name']) else 'Hẻm không tên'
    
    c = {
        'id': cid, 'u': u, 'v': v, 'do_dai': do_dai,
        'thoi_gian': thoi_gian, 'la_hem': la_hem, 'goc': goc, 'ten': ten
    }
    do_thi[u].append(c)
    thong_tin_canh[cid] = c

print(f"Xây dựng thành công đồ thị gồm {len(thong_tin_canh)} cung đường có hướng.")

Xây dựng thành công đồ thị gồm 8315 cung đường có hướng.


### Phần 5: Thuật toán Yen's K-Shortest Path (Lấy Top 1)

In [5]:
def tim_duong_ngan_nhat(s, t, canh_truoc=None, nut_cam=None, canh_cam=None):
    nut_cam = nut_cam or set()
    canh_cam = canh_cam or set()
    if s == t: return []
    
    pq = []
    v_max = 50.0 / 3.6
    heapq.heappush(pq, (tinh_khoang_cach(s, t) / v_max, 0.0, s, canh_truoc))
    chi_phi_tot = {(s, canh_truoc): 0.0}
    vet = {}
    dich = None
    
    while pq:
        f, g, u, in_id = heapq.heappop(pq)
        if g > chi_phi_tot.get((u, in_id), float('inf')):
            continue
        if u == t:
            dich = (u, in_id)
            break
            
        goc_truoc = thong_tin_canh[in_id]['goc'] if in_id else None
        hem_truoc = thong_tin_canh[in_id]['la_hem'] if in_id else False
        
        for c in do_thi.get(u, []):
            if c['id'] in canh_cam or (c['v'] in nut_cam and c['v'] != t):
                continue
            if in_id and c['v'] == thong_tin_canh[in_id]['u'] and len(do_thi.get(u, [])) > 1:
                continue
                
            tre, _ = tinh_do_tre_re(goc_truoc, c['goc'], hem_truoc, c['la_hem'])
            phat_cut = BANG_DO_TRE.get('phat_hem_cut', 300.0) if (len(do_thi.get(c['v'], [])) == 0 and c['v'] != t) else 0.0
            g_moi = g + c['thoi_gian'] + tre + phat_cut
            tt_ke = (c['v'], c['id'])
            
            if g_moi < chi_phi_tot.get(tt_ke, float('inf')):
                chi_phi_tot[tt_ke] = g_moi
                vet[tt_ke] = ((u, in_id), c)
                f_moi = g_moi + tinh_khoang_cach(c['v'], t) / v_max
                heapq.heappush(pq, (f_moi, g_moi, c['v'], c['id']))
                
    if not dich: return None
    duong_di, cur = [], dich
    while cur in vet:
        cur, c = vet[cur]
        duong_di.append(c)
    duong_di.reverse()
    return duong_di

def tong_hop_chi_so(duong_di):
    tong_m = sum(c['do_dai'] for c in duong_di)
    hem_m = sum(c['do_dai'] for c in duong_di if c['la_hem'])
    tong_s = 0.0
    so_lan_re = 0
    goc_truoc, hem_truoc = None, False
    
    for c in duong_di:
        tre, hanh_dong = tinh_do_tre_re(goc_truoc, c['goc'], hem_truoc, c['la_hem'])
        tong_s += c['thoi_gian'] + tre
        if goc_truoc is not None and hanh_dong in ['Rẽ trái', 'Rẽ phải', 'Quay đầu']:
            so_lan_re += 1
        goc_truoc, hem_truoc = c['goc'], c['la_hem']
        
    return {
        'quang_duong_km': tong_m / 1000.0,
        'thoi_gian_phut': tong_s / 60.0,
        'so_lan_re': so_lan_re,
        'ty_le_hem': (hem_m / tong_m * 100.0) if tong_m > 0 else 0.0
    }

def thuat_toan_yen_top1(s, t, k=3):
    p0 = tim_duong_ngan_nhat(s, t)
    if not p0: return None
    A = [p0]
    B = []
    
    for _ in range(1, k):
        prev = A[-1]
        nodes = [prev[0]['u']] + [c['v'] for c in prev]
        for i in range(len(prev)):
            spur_node = nodes[i]
            root = prev[:i]
            in_id = root[-1]['id'] if root else None
            canh_cam = {p[i]['id'] for p in A if len(p) > i and [c['id'] for c in p[:i]] == [c['id'] for c in root]}
            nut_cam = set(nodes[:i])
            spur = tim_duong_ngan_nhat(spur_node, t, in_id, nut_cam, canh_cam)
            if spur is not None:
                cand = root + spur
                cand_nodes = [cand[0]['u']] + [c['v'] for c in cand]
                if len(cand_nodes) == len(set(cand_nodes)):
                    cs = tong_hop_chi_so(cand)
                    heapq.heappush(B, (cs['thoi_gian_phut'] * 60.0, tuple(c['id'] for c in cand), cand))
        a_keys = {tuple(c['id'] for c in p) for p in A}
        while B:
            _, key, cand = heapq.heappop(B)
            if key not in a_keys:
                A.append(cand)
                break
    return tong_hop_chi_so(A[0])

### Phần 6: Chạy kết quả Top 1 từ file Excel

In [6]:
bang_ket_qua = []

for _, r in df_od_5.iterrows():
    s, t = int(r['source']), int(r['target'])
    ten = f"{r['origin_road']} -> {r['destination_road']}"
    
    kq = thuat_toan_yen_top1(s, t, k=3)
    if not kq: continue
    
    bang_ket_qua.append({
        "Mã OD": r['od_id'],
        "Tuyến đường": ten,
        "Tổng quãng đường đi (km)": round(kq['quang_duong_km'], 2),
        "Thời gian di chuyển (phút)": round(kq['thoi_gian_phut'], 2),
        "Tổng số lần rẽ": kq['so_lan_re'],
        "Tỷ lệ đi hẻm (%)": round(kq['ty_le_hem'], 1)
    })

df_ket_qua = pd.DataFrame(bang_ket_qua)
df_ket_qua

,Mã OD,Tuyến đường,Tổng quãng đường đi (km),Thời gian di chuyển (phút),Tổng số lần rẽ,Tỷ lệ đi hẻm (%)
0,OD06,Xô Viết Nghệ Tĩnh -> Nguyễn Gia Trí,1.02,7.09,4,48.5
1,OD11,Xô Viết Nghệ Tĩnh -> Ung Văn Khiêm,0.42,2.83,1,100.0
2,OD14,Ung Văn Khiêm -> Xô Viết Nghệ Tĩnh,0.42,3.69,4,81.3
3,OD25,Ung Văn Khiêm -> Điện Biên Phủ,2.29,13.05,5,7.1
4,OD28,Ung Văn Khiêm -> Nguyễn Gia Trí,0.19,1.66,1,100.0
